In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
from collections import defaultdict

#### 파일 불러오기

In [3]:
files = [
    "2019-Oct.csv",
    "2019-Nov.csv"
]

chunksize = 500000

#### 1.1 플래그 변수 추가

In [4]:
# 필요한 컬럼만 불러오기
usecols = [
    "event_time", "event_type", "product_id", "category_code",
    "brand", "price", "user_id", "user_session"
]

def add_missing_flags(chunk):
    df = chunk.copy()

    # 문자열 공백 제거 + 빈 문자열 결측 처리
    for col in ["event_type", "category_code", "brand", "user_session"]:
        if col in df.columns:
            df[col] = df[col].astype("string").str.strip()
            df[col] = df[col].replace("", pd.NA)

    # 1. 결측 플래그 생성
    df["brand_missing"] = df["brand"].isna().astype("int")
    df["category_missing"] = df["category_code"].isna().astype("int")

    # price가 결측, 0 이하일때 변수 invalid 생성
    df["price_invalid"] = ((df["price"].isna()) | (df["price"] <= 0)).astype("int")

    # 2. 결측 대체 컬럼 생성
    df["brand_filled"] = df["brand"].fillna("unknown_brand")
    df["category_filled"] = df["category_code"].fillna("unknown_category")

    # 3. 스마트폰 여부 파생
    df["is_smartphone"] = (
        df["category_filled"]
        .str.contains(r"smartphone", case=False, na=False)
        .astype("int")
    )

    return df

summary_list = []

for file in files:
    file_name = Path(file).name

    total_rows = 0
    brand_missing_cnt = 0
    category_missing_cnt = 0
    price_invalid_cnt = 0
    smartphone_rows = 0

    reader = pd.read_csv(
        file,
        usecols=usecols,
        chunksize=chunksize
    )

    for chunk in reader:
        df = add_missing_flags(chunk)

        total_rows += len(df)
        brand_missing_cnt += df["brand_missing"].sum()
        category_missing_cnt += df["category_missing"].sum()
        price_invalid_cnt += df["price_invalid"].sum()
        smartphone_rows += df["is_smartphone"].sum()

    summary_list.append({
        "file": file_name,
        "total_rows": total_rows,
        "brand 결측 수": int(brand_missing_cnt),
        "brand 결측 비율": round(brand_missing_cnt / total_rows * 100, 4),
        "category 결측 수": int(category_missing_cnt),
        "category 결측 비율": round(category_missing_cnt / total_rows * 100, 4),
        "price 0 수": int(price_invalid_cnt),
        "price 0 비율": round(price_invalid_cnt / total_rows * 100, 4),
        "smartphone_rows": int(smartphone_rows),
        "smartphone 비율": round(smartphone_rows / total_rows * 100, 4)
    })

summary_df = pd.DataFrame(summary_list)
display(summary_df)

,file,total_rows,brand 결측 수,brand 결측 비율,category 결측 수,category 결측 비율,price 0 수,price 0 비율,smartphone_rows,smartphone 비율
0,2019-Oct.csv,42448764,6117080,14.4105,13515609,31.8398,68673,0.1618,11507231,27.1085
1,2019-Nov.csv,67501979,9224078,13.6649,21898171,32.4408,188088,0.2786,16375000,24.2585


해석
- brand와 category_code결측 비율이 의미 있는 비중
  - category_code 결측은 3/1 정도로 있어 삭제 X
- price가 0인 비율은 적지만, 비정상 데이터일 가능성이 높아 따로 확인

결과
- brand, category_code는 결측 여부를 플래그로 보존 (brand_missing, category_missing)
- 결측을 'unknown_brand', 'category_brand'로 대체한 컬럼 추가 생성(brand_filled, category_filled)
- price = 0인 칼럼 'price_invalid' 플래그 생성해 비교

<br>
=> 결측과 이상치를 제거하지 않고, 원본 + 결측 채운 칼럼 + 플래그 칼럼을 같이 유지한 방향으로 진행

#### 1.2 smartphone 통계

In [5]:
smartphone_summary_list = []

for file in files:
    file_name = Path(file).name

    smartphone_total = 0
    brand_missing_cnt = 0
    category_missing_cnt = 0
    price_invalid_cnt = 0

    reader = pd.read_csv(
        file,
        usecols=usecols,
        chunksize=chunksize
    )

    for chunk in reader:
        df = add_missing_flags(chunk)
        phone_df = df[df["is_smartphone"] == 1]

        smartphone_total += len(phone_df)
        brand_missing_cnt += phone_df["brand_missing"].sum()
        category_missing_cnt += phone_df["category_missing"].sum()
        price_invalid_cnt += phone_df["price_invalid"].sum()

    smartphone_summary_list.append({
        "file": file_name,
        "smartphone_total_rows": int(smartphone_total),
        "brand 결측 수": int(brand_missing_cnt),
        "brand 결측 비율": round((brand_missing_cnt / smartphone_total * 100), 4) if smartphone_total > 0 else 0,
        "price_invalid 수": int(price_invalid_cnt),
        "price_invalid 비율": round((price_invalid_cnt / smartphone_total * 100), 4) if smartphone_total > 0 else 0
    })

smartphone_summary_df = pd.DataFrame(smartphone_summary_list)
display(smartphone_summary_df)

,file,smartphone_total_rows,brand 결측 수,brand 결측 비율,price_invalid 수,price_invalid 비율
0,2019-Oct.csv,11507231,21910,0.1904,4804,0.0417
1,2019-Nov.csv,16375000,21418,0.1308,6506,0.0397


해석
- 스마트폰은 brand 결측이 매우 낮음 (0.2%이하)
- 가격이 0인 비율도 매우 낮음 (0.05%이하)
- 스마트폰은 전체 데이터에 비에 안정적(크게 결측치들에 대해 영향을 받지x)

결과
- 왜곡이 적지만, 스마트폰도 분석에서 플래그는 유지(틀 유지)
- 추가로 구매 행동 분석 진행

=> 전체에 비에 스마트폰은 안정적이지만, 결측 여부에 따른 전환 확인 후 진행

#### 2.1 session-item 기준 집계 테이블

In [ ]:
def make_session_item_chunk(df):
    temp = df.copy()

    # 이벤트 더미화
    temp["view_yn"] = (temp["event_type"] == "view").astype("int")
    temp["cart_yn"] = (temp["event_type"] == "cart").astype("int")
    temp["purchase_yn"] = (temp["event_type"] == "purchase").astype("int")

    # session-item 기준 집계
    grouped = temp.groupby(["user_session", "product_id"], as_index=False).agg(
        view_yn=("view_yn", "max"),
        cart_yn=("cart_yn", "max"),
        purchase_yn=("purchase_yn", "max"),

        brand_missing=("brand_missing", "max"),
        category_missing=("category_missing", "max"),
        price_invalid=("price_invalid", "max"),

        is_smartphone=("is_smartphone", "max")
    )

    return grouped


def update_summary(summary_dict, df, group_col, file_name):
    for group_value, sub in df.groupby(group_col):
        key = (file_name, group_col, int(group_value))

        summary_dict[key]["session_item_count"] += len(sub)
        summary_dict[key]["view_count"] += int(sub["view_yn"].sum())
        summary_dict[key]["cart_count"] += int(sub["cart_yn"].sum())
        summary_dict[key]["purchase_count"] += int(sub["purchase_yn"].sum())


# 누적 요약 저장용
summary_dict = defaultdict(lambda: {
    "session_item_count": 0,
    "view_count": 0,
    "cart_count": 0,
    "purchase_count": 0
})

# 파일별 전체 session-item 수만 따로 확인
overall_summary = []

for file in files:
    file_name = Path(file).name
    print(f"\n===== {file_name} 처리 중 =====")

    total_session_items = 0
    total_smartphone_session_items = 0

    reader = pd.read_csv(
        file,
        usecols=usecols,
        chunksize=chunksize
    )

    for i, chunk in enumerate(reader, start=1):
        df = add_missing_flags(chunk)
        session_item_chunk = make_session_item_chunk(df)

        total_session_items += len(session_item_chunk)
        total_smartphone_session_items += int(session_item_chunk["is_smartphone"].sum())

        # 전체 기준 비교
        update_summary(summary_dict, session_item_chunk, "brand_missing", file_name)
        update_summary(summary_dict, session_item_chunk, "category_missing", file_name)
        update_summary(summary_dict, session_item_chunk, "price_invalid", file_name)

        if i % 10 == 0:
            print(f"  - chunk {i}개 처리 완료")

    overall_summary.append({
        "file": file_name,
        "session_item_count": total_session_items,
        "smartphone_session_item_count": total_smartphone_session_items,
        "smartphone_pct": round(total_smartphone_session_items / total_session_items * 100, 4)
        if total_session_items > 0 else 0
    })

overall_summary_df = pd.DataFrame(overall_summary)
display(overall_summary_df)


===== 2019-Oct.csv 처리 중 =====
  - chunk 10개 처리 완료
  - chunk 20개 처리 완료
  - chunk 30개 처리 완료
  - chunk 40개 처리 완료
  - chunk 50개 처리 완료
  - chunk 60개 처리 완료
  - chunk 70개 처리 완료
  - chunk 80개 처리 완료

===== 2019-Nov.csv 처리 중 =====
  - chunk 10개 처리 완료
  - chunk 20개 처리 완료
  - chunk 30개 처리 완료
  - chunk 40개 처리 완료
  - chunk 50개 처리 완료
  - chunk 60개 처리 완료
  - chunk 70개 처리 완료
  - chunk 80개 처리 완료
  - chunk 90개 처리 완료
  - chunk 100개 처리 완료
  - chunk 110개 처리 완료
  - chunk 120개 처리 완료
  - chunk 130개 처리 완료


,file,session_item_count,smartphone_session_item_count,smartphone_pct
0,2019-Oct.csv,27917253,7102649,25.4418
1,2019-Nov.csv,42378839,9675448,22.8308


해석
- user_session + product_id 기준으로 진행
  - session_item_count로 session_item 테이블 생성
- session_item 기준으로 진행함으로써, 한 세션 특정 상품에 대해 조회, 장바구니, 구매가 발생했는지 파악
  - row로 진행시, 같은 상품에 대한 이벤트가 반복되어서 비교시 영향이 있음

결과
- 결측 이상치는 row 기준이 아닌 session-item 기준으로 진행
- 이벤트 전환율 비교 또한 session-item로 진행 

=> 결측 이상치 영향 비교를 row가 아닌 session-item로

In [7]:
summary_rows = []

for (file_name, group_col, group_value), vals in summary_dict.items():
    session_cnt = vals["session_item_count"]
    view_cnt = vals["view_count"]
    cart_cnt = vals["cart_count"]
    purchase_cnt = vals["purchase_count"]

    summary_rows.append({
        "file": file_name,
        "group_col": group_col,
        "group_value": group_value,
        "session_item_count": session_cnt,
        "view_count": view_cnt,
        "cart_count": cart_cnt,
        "purchase_count": purchase_cnt,
        "view_rate": round(view_cnt / session_cnt * 100, 4) if session_cnt > 0 else 0,
        "cart_rate": round(cart_cnt / session_cnt * 100, 4) if session_cnt > 0 else 0,
        "purchase_rate": round(purchase_cnt / session_cnt * 100, 4) if session_cnt > 0 else 0,
        "cart_to_purchase_rate": round(purchase_cnt / cart_cnt * 100, 4) if cart_cnt > 0 else 0
    })

session_item_summary_df = pd.DataFrame(summary_rows).sort_values(
    ["file", "group_col", "group_value"]
).reset_index(drop=True)

display(session_item_summary_df)

,file,group_col,group_value,session_item_count,view_count,cart_count,purchase_count,view_rate,cart_rate,purchase_rate,cart_to_purchase_rate
0,2019-Nov.csv,brand_missing,0,36007109,35980051,1882395,789134,99.9249,5.2278,2.1916,41.9218
1,2019-Nov.csv,brand_missing,1,6371730,6369754,186488,70242,99.9690,2.9268,1.1024,37.6657
2,2019-Nov.csv,category_missing,0,27886163,27863246,1481396,636673,99.9178,5.3123,2.2831,42.9779
3,2019-Nov.csv,category_missing,1,14492676,14486559,587487,222703,99.9578,4.0537,1.5367,37.9077
4,2019-Nov.csv,price_invalid,0,42244269,42215249,2065456,858623,99.9313,4.8893,2.0325,41.5706
5,2019-Nov.csv,price_invalid,1,134570,134556,3427,753,99.9896,2.5466,0.5596,21.9726
6,2019-Oct.csv,brand_missing,0,23493234,23488618,616757,635425,99.9804,2.6253,2.7047,103.0268
7,2019-Oct.csv,brand_missing,1,4424019,4423773,12819,55465,99.9944,0.2898,1.2537,432.6781
8,2019-Oct.csv,category_missing,0,18465190,18461305,555512,526640,99.9790,3.0084,2.8521,94.8026
9,2019-Oct.csv,category_missing,1,9452063,9451086,74064,164250,99.9897,0.7836,1.7377,221.7677


#### 2.2 스마트폰만

In [8]:
smartphone_summary_dict = defaultdict(lambda: {
    "session_item_count": 0,
    "view_count": 0,
    "cart_count": 0,
    "purchase_count": 0
})

for file in files:
    file_name = Path(file).name
    print(f"\n===== {file_name} 스마트폰 subset 처리 중 =====")

    reader = pd.read_csv(
        file,
        usecols=usecols,
        chunksize=chunksize
    )

    for i, chunk in enumerate(reader, start=1):
        df = add_missing_flags(chunk)
        session_item_chunk = make_session_item_chunk(df)

        phone_df = session_item_chunk[session_item_chunk["is_smartphone"] == 1]

        update_summary(smartphone_summary_dict, phone_df, "brand_missing", file_name)
        update_summary(smartphone_summary_dict, phone_df, "price_invalid", file_name)

        if i % 10 == 0:
            print(f"  - chunk {i}개 처리 완료")


===== 2019-Oct.csv 스마트폰 subset 처리 중 =====
  - chunk 10개 처리 완료
  - chunk 20개 처리 완료
  - chunk 30개 처리 완료
  - chunk 40개 처리 완료
  - chunk 50개 처리 완료
  - chunk 60개 처리 완료
  - chunk 70개 처리 완료
  - chunk 80개 처리 완료

===== 2019-Nov.csv 스마트폰 subset 처리 중 =====
  - chunk 10개 처리 완료
  - chunk 20개 처리 완료
  - chunk 30개 처리 완료
  - chunk 40개 처리 완료
  - chunk 50개 처리 완료
  - chunk 60개 처리 완료
  - chunk 70개 처리 완료
  - chunk 80개 처리 완료
  - chunk 90개 처리 완료
  - chunk 100개 처리 완료
  - chunk 110개 처리 완료
  - chunk 120개 처리 완료
  - chunk 130개 처리 완료


In [9]:
smartphone_rows = []

for (file_name, group_col, group_value), vals in smartphone_summary_dict.items():
    session_cnt = vals["session_item_count"]
    view_cnt = vals["view_count"]
    cart_cnt = vals["cart_count"]
    purchase_cnt = vals["purchase_count"]

    smartphone_rows.append({
        "file": file_name,
        "group_col": group_col,
        "group_value": group_value,
        "session_item_count": session_cnt,
        "view_count": view_cnt,
        "cart_count": cart_cnt,
        "purchase_count": purchase_cnt,
        "view_rate": round(view_cnt / session_cnt * 100, 4) if session_cnt > 0 else 0,
        "cart_rate": round(cart_cnt / session_cnt * 100, 4) if session_cnt > 0 else 0,
        "purchase_rate": round(purchase_cnt / session_cnt * 100, 4) if session_cnt > 0 else 0,
        "cart_to_purchase_rate": round(purchase_cnt / cart_cnt * 100, 4) if cart_cnt > 0 else 0
    })

smartphone_session_item_summary_df = pd.DataFrame(smartphone_rows).sort_values(
    ["file", "group_col", "group_value"]
).reset_index(drop=True)

display(smartphone_session_item_summary_df)

,file,group_col,group_value,session_item_count,view_count,cart_count,purchase_count,view_rate,cart_rate,purchase_rate,cart_to_purchase_rate
0,2019-Nov.csv,brand_missing,0,9662193,9649448,759473,352348,99.8681,7.8603,3.6467,46.3937
1,2019-Nov.csv,brand_missing,1,13255,13249,418,147,99.9547,3.1535,1.1090,35.1675
2,2019-Nov.csv,price_invalid,0,9671024,9658273,759821,352474,99.8682,7.8567,3.6446,46.3891
3,2019-Nov.csv,price_invalid,1,4424,4424,70,21,100.0000,1.5823,0.4747,30.0000
4,2019-Oct.csv,brand_missing,0,7088438,7085940,368262,308836,99.9648,5.1952,4.3569,83.8631
5,2019-Oct.csv,brand_missing,1,14211,14211,267,406,100.0000,1.8788,2.8569,152.0599
6,2019-Oct.csv,price_invalid,0,7099408,7096910,368518,309203,99.9648,5.1908,4.3553,83.9044
7,2019-Oct.csv,price_invalid,1,3241,3241,11,39,100.0000,0.3394,1.2033,354.5455


#### 3.1 결측/이상치 여부별 전환율 비교

In [ ]:
step3_compare_df = (
    session_item_summary_df[
        ["file", "group_col", "group_value", "session_item_count", "cart_rate", "purchase_rate"]
    ]
    .sort_values(["file", "group_col", "group_value"])
    .reset_index(drop=True)
)

display(step3_compare_df)

,file,group_col,group_value,session_item_count,cart_rate,purchase_rate
0,2019-Nov.csv,brand_missing,0,36007109,5.2278,2.1916
1,2019-Nov.csv,brand_missing,1,6371730,2.9268,1.1024
2,2019-Nov.csv,category_missing,0,27886163,5.3123,2.2831
3,2019-Nov.csv,category_missing,1,14492676,4.0537,1.5367
4,2019-Nov.csv,price_invalid,0,42244269,4.8893,2.0325
5,2019-Nov.csv,price_invalid,1,134570,2.5466,0.5596
6,2019-Oct.csv,brand_missing,0,23493234,2.6253,2.7047
7,2019-Oct.csv,brand_missing,1,4424019,0.2898,1.2537
8,2019-Oct.csv,category_missing,0,18465190,3.0084,2.8521
9,2019-Oct.csv,category_missing,1,9452063,0.7836,1.7377


해석
- 결측/이상치 그룹(1)이 정상 그룹(0)보다 낮게 나옴
- 단순 누락이 아닌 행동 특성과 연관되어 있을 가능성 있음

결과
- 분석 변수 유지

=> 행동 변수 설명하는 걸 유지

In [ ]:
# 스마트폰만

step3_phone_compare_df = (
    smartphone_session_item_summary_df[
        ["file", "group_col", "group_value", "session_item_count", "cart_rate", "purchase_rate"]
    ]
    .sort_values(["file", "group_col", "group_value"])
    .reset_index(drop=True)
)

display(step3_phone_compare_df)

,file,group_col,group_value,session_item_count,cart_rate,purchase_rate
0,2019-Nov.csv,brand_missing,0,9662193,7.8603,3.6467
1,2019-Nov.csv,brand_missing,1,13255,3.1535,1.1090
2,2019-Nov.csv,price_invalid,0,9671024,7.8567,3.6446
3,2019-Nov.csv,price_invalid,1,4424,1.5823,0.4747
4,2019-Oct.csv,brand_missing,0,7088438,5.1952,4.3569
5,2019-Oct.csv,brand_missing,1,14211,1.8788,2.8569
6,2019-Oct.csv,price_invalid,0,7099408,5.1908,4.3553
7,2019-Oct.csv,price_invalid,1,3241,0.3394,1.2033


In [ ]:
# 그룹별 purchase_rate 차이

purchase_compare = (
    session_item_summary_df.pivot_table(
        index=["file", "group_col"],
        columns="group_value",
        values="purchase_rate"
    )
    .reset_index()
)

purchase_compare.columns = ["file", "group_col", "purchase_rate_0", "purchase_rate_1"]
purchase_compare["diff_1_minus_0"] = (
    purchase_compare["purchase_rate_1"] - purchase_compare["purchase_rate_0"]
)

display(purchase_compare)

,file,group_col,purchase_rate_0,purchase_rate_1,diff_1_minus_0
0,2019-Nov.csv,brand_missing,2.1916,1.1024,-1.0892
1,2019-Nov.csv,category_missing,2.2831,1.5367,-0.7464
2,2019-Nov.csv,price_invalid,2.0325,0.5596,-1.4729
3,2019-Oct.csv,brand_missing,2.7047,1.2537,-1.4510
4,2019-Oct.csv,category_missing,2.8521,1.7377,-1.1144
5,2019-Oct.csv,price_invalid,2.4778,0.8057,-1.6721


해석
- 모두 diff_1_minus_0가 마이너스
- 결측/이상치가 있는 그룹(1)의 구매 전환율이 정상(0)보다 낮음

결과
- price_invalid는 행동분석에서는 유지, 매출/AOV 같은 금액 분석에는 제외

In [ ]:
# 그룹별 purchase_rate 차이 스마트폰

phone_purchase_compare = (
    smartphone_session_item_summary_df.pivot_table(
        index=["file", "group_col"],
        columns="group_value",
        values="purchase_rate"
    )
    .reset_index()
)

phone_purchase_compare.columns = ["file", "group_col", "purchase_rate_0", "purchase_rate_1"]
phone_purchase_compare["diff_1_minus_0"] = (
    phone_purchase_compare["purchase_rate_1"] - phone_purchase_compare["purchase_rate_0"]
)

display(phone_purchase_compare)

,file,group_col,purchase_rate_0,purchase_rate_1,diff_1_minus_0
0,2019-Nov.csv,brand_missing,3.6467,1.1090,-2.5377
1,2019-Nov.csv,price_invalid,3.6446,0.4747,-3.1699
2,2019-Oct.csv,brand_missing,4.3569,2.8569,-1.5000
3,2019-Oct.csv,price_invalid,4.3553,1.2033,-3.1520


해석
- 전체와 같이 결측/이상치가 정상보다 낮음
  - 스마트폰이 더 뚜렷하게 보임

결과
- 분석 결과 비율이 낮다고 제거하면 X
- 의미가 있어 남겨두어야 함

=> 스마트폰도 결측/이상치 플래그를 유지하는 전처리 진행

#### 4. 교차검증

In [ ]:
def update_cross_summary(summary_dict, df, file_name, col1, col2):
    grouped = (
        df.groupby([col1, col2], dropna=False)
          .agg(
              session_item_count=("product_id", "size"),
              cart_count=("cart_yn", "sum"),
              purchase_count=("purchase_yn", "sum")
          )
          .reset_index()
    )

    for _, row in grouped.iterrows():
        key = (file_name, col1, col2, row[col1], row[col2])
        summary_dict[key]["session_item_count"] += int(row["session_item_count"])
        summary_dict[key]["cart_count"] += int(row["cart_count"])
        summary_dict[key]["purchase_count"] += int(row["purchase_count"])

In [ ]:
# 교차 검증

cross_pairs = [
    ("brand_missing", "is_smartphone"),
    ("category_missing", "is_smartphone"),
    ("price_invalid", "is_smartphone"),
    ("brand_missing", "category_missing")
]

cross_summary_dict = defaultdict(lambda: {
    "session_item_count": 0,
    "cart_count": 0,
    "purchase_count": 0
})

for file in files:
    file_name = Path(file).name
    print(f"\n===== {file_name} 교차 검증 처리 중 =====")

    reader = pd.read_csv(
        file,
        usecols=usecols,
        chunksize=chunksize
    )

    for i, chunk in enumerate(reader, start=1):
        df = add_missing_flags(chunk)
        session_item_chunk = make_session_item_chunk(df)

        for col1, col2 in cross_pairs:
            update_cross_summary(cross_summary_dict, session_item_chunk, file_name, col1, col2)

        if i % 10 == 0:
            print(f"{file_name}: {i}개 chunk 처리 완료")


===== 2019-Oct.csv 교차 검증 처리 중 =====
2019-Oct.csv: 10개 chunk 처리 완료
2019-Oct.csv: 20개 chunk 처리 완료
2019-Oct.csv: 30개 chunk 처리 완료
2019-Oct.csv: 40개 chunk 처리 완료
2019-Oct.csv: 50개 chunk 처리 완료
2019-Oct.csv: 60개 chunk 처리 완료
2019-Oct.csv: 70개 chunk 처리 완료
2019-Oct.csv: 80개 chunk 처리 완료

===== 2019-Nov.csv 교차 검증 처리 중 =====
2019-Nov.csv: 10개 chunk 처리 완료
2019-Nov.csv: 20개 chunk 처리 완료
2019-Nov.csv: 30개 chunk 처리 완료
2019-Nov.csv: 40개 chunk 처리 완료
2019-Nov.csv: 50개 chunk 처리 완료
2019-Nov.csv: 60개 chunk 처리 완료
2019-Nov.csv: 70개 chunk 처리 완료
2019-Nov.csv: 80개 chunk 처리 완료
2019-Nov.csv: 90개 chunk 처리 완료
2019-Nov.csv: 100개 chunk 처리 완료
2019-Nov.csv: 110개 chunk 처리 완료
2019-Nov.csv: 120개 chunk 처리 완료
2019-Nov.csv: 130개 chunk 처리 완료


In [16]:
# 전체 결과표
cross_rows = []

for (file_name, col1, col2, val1, val2), vals in cross_summary_dict.items():
    session_cnt = vals["session_item_count"]
    cart_cnt = vals["cart_count"]
    purchase_cnt = vals["purchase_count"]

    cross_rows.append({
        "file": file_name,
        "col1": col1,
        "col2": col2,
        "value1": val1,
        "value2": val2,
        "session_item_count": session_cnt,
        "cart_count": cart_cnt,
        "purchase_count": purchase_cnt,
        "cart_rate": round(cart_cnt / session_cnt * 100, 4) if session_cnt > 0 else 0,
        "purchase_rate": round(purchase_cnt / session_cnt * 100, 4) if session_cnt > 0 else 0
    })

cross_summary_df = pd.DataFrame(cross_rows).sort_values(
    ["file", "col1", "col2", "value1", "value2"]
).reset_index(drop=True)

display(cross_summary_df)

,file,col1,col2,value1,value2,session_item_count,cart_count,purchase_count,cart_rate,purchase_rate
0,2019-Nov.csv,brand_missing,category_missing,0,0,25489947,1419485,614190,5.5688,2.4095
1,2019-Nov.csv,brand_missing,category_missing,0,1,10517162,462910,174944,4.4015,1.6634
2,2019-Nov.csv,brand_missing,category_missing,1,0,2396216,61911,22483,2.5837,0.9383
3,2019-Nov.csv,brand_missing,category_missing,1,1,3975514,124577,47759,3.1336,1.2013
4,2019-Nov.csv,brand_missing,is_smartphone,0,0,26344916,1122922,436786,4.2624,1.6580
5,2019-Nov.csv,brand_missing,is_smartphone,0,1,9662193,759473,352348,7.8603,3.6467
6,2019-Nov.csv,brand_missing,is_smartphone,1,0,6358475,186070,70095,2.9263,1.1024
7,2019-Nov.csv,brand_missing,is_smartphone,1,1,13255,418,147,3.1535,1.1090
8,2019-Nov.csv,category_missing,is_smartphone,0,0,18210715,721505,284178,3.9620,1.5605
9,2019-Nov.csv,category_missing,is_smartphone,0,1,9675448,759891,352495,7.8538,3.6432


해석
- 결측/이상치가 특정 부분에 몰려있는지, 다른 결측과 같이 나타는지 교차 검증
  - 특정 편중 문제인지, 구조적 문제인지 확인

In [17]:
# brand_missing × is_smartphone
display(
    cross_summary_df[
        (cross_summary_df["col1"] == "brand_missing") &
        (cross_summary_df["col2"] == "is_smartphone")
    ]
)

# category_missing × is_smartphone
display(
    cross_summary_df[
        (cross_summary_df["col1"] == "category_missing") &
        (cross_summary_df["col2"] == "is_smartphone")
    ]
)

# price_invalid × is_smartphone
display(
    cross_summary_df[
        (cross_summary_df["col1"] == "price_invalid") &
        (cross_summary_df["col2"] == "is_smartphone")
    ]
)

# brand_missing × category_missing
display(
    cross_summary_df[
        (cross_summary_df["col1"] == "brand_missing") &
        (cross_summary_df["col2"] == "category_missing")
    ]
)

,file,col1,col2,value1,value2,session_item_count,cart_count,purchase_count,cart_rate,purchase_rate
4,2019-Nov.csv,brand_missing,is_smartphone,0,0,26344916,1122922,436786,4.2624,1.6580
5,2019-Nov.csv,brand_missing,is_smartphone,0,1,9662193,759473,352348,7.8603,3.6467
6,2019-Nov.csv,brand_missing,is_smartphone,1,0,6358475,186070,70095,2.9263,1.1024
7,2019-Nov.csv,brand_missing,is_smartphone,1,1,13255,418,147,3.1535,1.1090
19,2019-Oct.csv,brand_missing,is_smartphone,0,0,16404796,248495,326589,1.5148,1.9908
20,2019-Oct.csv,brand_missing,is_smartphone,0,1,7088438,368262,308836,5.1952,4.3569
21,2019-Oct.csv,brand_missing,is_smartphone,1,0,4409808,12552,55059,0.2846,1.2486
22,2019-Oct.csv,brand_missing,is_smartphone,1,1,14211,267,406,1.8788,2.8569


,file,col1,col2,value1,value2,session_item_count,cart_count,purchase_count,cart_rate,purchase_rate
8,2019-Nov.csv,category_missing,is_smartphone,0,0,18210715,721505,284178,3.9620,1.5605
9,2019-Nov.csv,category_missing,is_smartphone,0,1,9675448,759891,352495,7.8538,3.6432
10,2019-Nov.csv,category_missing,is_smartphone,1,0,14492676,587487,222703,4.0537,1.5367
23,2019-Oct.csv,category_missing,is_smartphone,0,0,11362541,186983,217398,1.6456,1.9133
24,2019-Oct.csv,category_missing,is_smartphone,0,1,7102649,368529,309242,5.1886,4.3539
25,2019-Oct.csv,category_missing,is_smartphone,1,0,9452063,74064,164250,0.7836,1.7377


,file,col1,col2,value1,value2,session_item_count,cart_count,purchase_count,cart_rate,purchase_rate
11,2019-Nov.csv,price_invalid,is_smartphone,0,0,32573245,1305635,506149,4.0083,1.5539
12,2019-Nov.csv,price_invalid,is_smartphone,0,1,9671024,759821,352474,7.8567,3.6446
13,2019-Nov.csv,price_invalid,is_smartphone,1,0,130146,3357,732,2.5794,0.5624
14,2019-Nov.csv,price_invalid,is_smartphone,1,1,4424,70,21,1.5823,0.4747
26,2019-Oct.csv,price_invalid,is_smartphone,0,0,20766959,260956,381277,1.2566,1.8360
27,2019-Oct.csv,price_invalid,is_smartphone,0,1,7099408,368518,309203,5.1908,4.3553
28,2019-Oct.csv,price_invalid,is_smartphone,1,0,47645,91,371,0.1910,0.7787
29,2019-Oct.csv,price_invalid,is_smartphone,1,1,3241,11,39,0.3394,1.2033


,file,col1,col2,value1,value2,session_item_count,cart_count,purchase_count,cart_rate,purchase_rate
0,2019-Nov.csv,brand_missing,category_missing,0,0,25489947,1419485,614190,5.5688,2.4095
1,2019-Nov.csv,brand_missing,category_missing,0,1,10517162,462910,174944,4.4015,1.6634
2,2019-Nov.csv,brand_missing,category_missing,1,0,2396216,61911,22483,2.5837,0.9383
3,2019-Nov.csv,brand_missing,category_missing,1,1,3975514,124577,47759,3.1336,1.2013
15,2019-Oct.csv,brand_missing,category_missing,0,0,16779196,547806,507747,3.2648,3.0261
16,2019-Oct.csv,brand_missing,category_missing,0,1,6714038,68951,127678,1.0270,1.9017
17,2019-Oct.csv,brand_missing,category_missing,1,0,1685994,7706,18893,0.4571,1.1206
18,2019-Oct.csv,brand_missing,category_missing,1,1,2738025,5113,36572,0.1867,1.3357


brand_missing × is_smartphone
- (스마트폰) × (정상 브랜드)가 구매 전환율이 가장 높음
- 스마트폰 여부와 상관없이 결측 브랜드 구매 전환율이 낮음
- 스마트폰은 구매가 비교적 높은 상품
  - 특히, 브랜드가 정상일 때 높음
- 브랜드 결측이 있는 것은 전환율이 낮음
  - brand_missing 단순 누락이 아닐수있음
    - 전환
    - 품질 신호

=> brand_missing은 삭제보다 분석 변수로 보존 <br>

=========================================

category_missing × is_smartphone
- (스마트폰) × (정상 카테고리)가 구매 전환율이 가장 높음
- 비스마트폰은 카테고리 여부에 따른 차이가 상대적으로 적음

=> category_code 결측은 비율도 크고 전환 차이도 존재. <br>
=> 삭제보다 unknown 범주 + 플래그 유지

=========================================

price_invalid × is_smartphone
- 정상 가격 그룹이 스마트폰 여부 상관없이 구매 전환율이 높음
- 가격이 0인 그룹은 모두 전환율이 많이 낮음
  - 가격 이상치는 전반적으로 전환율이 낮음
  - 데이터 오류 가능성이 높음

=> price_invalid 유지, 퍼널/행동에는 session-item 삭제X, 다만 금액 기반은 제외

=========================================

brand_missing × category_missing
- 정상 브랜드가 대체적으로 높음
- 결측 브랜드만 나타날 경우가 가장 낮음
- 브랜드와 카테고리 결측은 개별적으로 다른 의미로 나타남

=> brand_missing, category_missing 변수 유지해서 진행

=========================================

##### 결론
- brand_missing, category_missing, price_invalid
  - 단순 누락값, 예외값이라기보다 session-item 기준 구매 전환 차이와 연결되는 정보
- 결측/이상치 그룹은 정상 그룹보다 일관되게 낮은 구매 전환율
  - 스마트폰 subset에서도 같은 방향의 패턴
- 물론, 결측/이상치가 값이 작아 더 크게 다가오는 것일수도 있음
- 행동이 분석에 반영될 수 있음

=> 결측/이상치를 삭제하지말고 진행